In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

# Imports

In [3]:
from harp.data.collect import collect
from harp.data.process import clean_df, group_by_reviewer, inject_rejected, encode

import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Data Pipeline

In [5]:
collect(5, "data/raw/samples")
clean_df(pd.read_csv("harp/data/raw/cms_2008_2010_samples.csv"))
group_by_reviewer(pd.read_csv("harp/data/raw/cms_2008_2010_samples_cleaned.csv"))

df = pd.read_csv("harp/data/raw/csm_2008_2010_samples_grouped.csv")

RANDOM_STATE = 42

df_train_val, df_test = train_test_split(
    df, 
    test_size=0.15, 
    random_state=RANDOM_STATE,
    shuffle=True
)

validation_size = 0.15 / 0.85
df_train, df_val = train_test_split(
    df_train_val, 
    test_size=validation_size, 
    random_state=RANDOM_STATE,
    shuffle=True
)

df_train = inject_rejected(df_train, 0.2)
df_val = inject_rejected(df_val, 0.2)
df_test = inject_rejected(df_test, 0.2)

os.makedirs("harp/data/raw/harp_dataset")
df_train.to_csv("harp/data/raw/harp_dataset/train.csv", index=False)
df_val.to_csv("harp/data/raw/harp_dataset/val.csv", index=False)
df_test.to_csv("harp/data/raw/harp_dataset/test.csv", index=False)

os.makedirs("harp/data/raw/harp_dataset_encoded")
df_train_encoded = encode(df_train, os.path.join("harp/data/raw/harp_dataset_encoded", "train.csv"))
df_val_encoded = encode(df_val, os.path.join("harp/data/raw/harp_dataset_encoded", "val.csv"))
df_test_encoded = encode(df_test, os.path.join("harp/data/raw/harp_dataset_encoded", "test.csv"))

Initial size: 332606
Final clean size: 326651


## Determine good cutoffs for reviewer via quartiles